In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway
import os

countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']

dfs = []
for c in countries:
    path = f'data/{c}_clean.csv'
    if os.path.exists(path):
        temp = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
        dfs.append(temp)

df_all = pd.concat(dfs)

monthly_t2m = df_all.groupby(['Country', pd.Grouper(freq='M')])['T2M'].mean().reset_index()

plt.figure(figsize=(12,6))
for country in df_all['Country'].unique():
    data = monthly_t2m[monthly_t2m['Country'] == country]
    plt.plot(data['Date'], data['T2M'], label=country)
plt.title('Monthly Average Temperature (T2M) by Country')
plt.legend()
plt.grid(True)
plt.show()

print(df_all.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).round(2))

plt.figure(figsize=(10,6))
sns.boxplot(x='Country', y='PRECTOTCORR', data=df_all)
plt.show()

print(df_all.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).round(3))

df_all['Extreme_Heat'] = df_all['T2M_MAX'] > 35
heat_per_year = df_all.groupby(['Country', df_all.index.year])['Extreme_Heat'].sum().reset_index()

sns.barplot(x='Date', y='Extreme_Heat', hue='Country', data=heat_per_year)
plt.xticks(rotation=45)
plt.show()

df_all['Dry_Day'] = df_all['PRECTOTCORR'] < 1
dry_per_year = df_all.groupby(['Country', df_all.index.year])['Dry_Day'].sum().reset_index()

sns.barplot(x='Date', y='Dry_Day', hue='Country', data=dry_per_year)
plt.xticks(rotation=45)
plt.show()

groups = [df_all[df_all['Country'] == c]['T2M'].dropna() for c in countries]
f_stat, p_value = f_oneway(*groups)
print("ANOVA p-value:", round(p_value, 4))

ranking = pd.DataFrame({
    'Country': countries,
    'Avg_T2M': df_all.groupby('Country')['T2M'].mean().values,
    'T2M_Std': df_all.groupby('Country')['T2M'].std().values,
    'Precip_Std': df_all.groupby('Country')['PRECTOTCORR'].std().values,
    'Extreme_Heat_Days_per_Year': heat_per_year.groupby('Country')['Extreme_Heat'].mean().values,
    'Dry_Days_per_Year': dry_per_year.groupby('Country')['Dry_Day'].mean().values
})

ranking['Vulnerability_Score'] = (ranking['T2M_Std'] + ranking['Precip_Std']*10 + 
                                  ranking['Extreme_Heat_Days_per_Year']*0.5 + 
                                  ranking['Dry_Days_per_Year']*0.3)

ranking = ranking.sort_values('Vulnerability_Score', ascending=False).reset_index(drop=True)
print(ranking)